In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent / 'src'))

In [ ]:
from typing import List

from data_handlers.mic_data_handler import MicDataHandler
from utils.evaluation_utils import EvaluationUtils
from utils.project_utils import ProjectUtils

import textwrap

import pandas as pd

In [ ]:
project_root: Path = ProjectUtils.get_project_root(); project_root

In [ ]:
mic_data_handler: MicDataHandler = MicDataHandler(project_root)

In [ ]:
def evaluate_model(model_names: List[str], 
                   data_fold_values: List[int], 
                   replacement_strategies: List[str], 
                   export_path_root: Path) -> None:
    
    pe_df = mic_data_handler.get_private_entities_df()
    id_column="itemid"
    text_column="text"
    class_column="intents"
    pe_column="text_pe_ontonotes5_ner-english-ontonotes-large"
    zero_entity_retain_text=True

    for model_name in model_names:

        for k in data_fold_values:
            
            sample_size = 4339 if k in [1, 4, 5] else 4338
            data_dir_path = Path(f'/home/ssaha/model-checkpoints/mic/mltc/{model_name}/additional-embeddings-none/sample-size-{sample_size}/data-fold-{k}')
            model_file_path = data_dir_path / 'learning-rate-5e-5' / 'max-epochs-25' / 'mini-batch-size-8' / 'best-model.pt'
            test_df = mic_data_handler.get_train_dev_test_datasetdict(k=k)["test"].to_pandas()
            pe_df_test = pe_df[pe_df['itemid'].isin(test_df['itemid'])]
            entity_counts_stat = pe_df_test['pe_count_total'].describe()
            entity_counts_stat = entity_counts_stat.astype(int)
            print()
            print(f"Statistics of private entities in test samples for data_fold={k}:\n{pd.DataFrame(entity_counts_stat).to_markdown()}")
            print()
            
            input_df = test_df.copy()            
            for replacement_strategy in replacement_strategies:
                print()
                print(textwrap.dedent(f"""
                    Evaluating for configuration:
                    -> model={model_name} 
                    -> data_fold={k}
                    -> replacement_strategy={replacement_strategy}
                """).strip())
                print()
                result = EvaluationUtils.redact_and_evaluate_for_mic_mltc(input_df=input_df,
                                                                          pe_df=pe_df,
                                                                          id_column=id_column,
                                                                          text_column=text_column,
                                                                          class_column=class_column,
                                                                          pe_column=pe_column,
                                                                          replacement_strategy=replacement_strategy,
                                                                          zero_entity_retain_text=zero_entity_retain_text,
                                                                          data_dir_path=data_dir_path,
                                                                          model_file_path=model_file_path)
                
                export_path_dir = export_path_root / f'{model_name}' / f'{replacement_strategy}'
                export_path_dir.mkdir(parents=True, exist_ok=True)
                export_path = export_path_dir / f'K{k}.txt'
                with open(export_path, 'w') as f:
                    f.write(result.detailed_results)
                
                print(export_path.read_text())
                print()

In [ ]:
model_names = ["xlm-roberta-large"] ## ["xlm-roberta-large", "bert-large-cased", "microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract"]
data_fold_values = [1] ## [1, 2, 3, 4, 5]
replacement_strategies = ["semantic_label_mask"] ## ["semantic_label_mask", "random_mask", "generic_mask"]
export_path_root = Path("/home/ssaha/Projects/mic_mltc_metrics")

evaluate_model(model_names, data_fold_values, replacement_strategies, export_path_root)